# Project8 v0.5 — 15.000 ảnh, fine-tune từ BEST v0.4

Upload gói train/val mới và best v0.4. Gói có 12.000 train + 2.500 val; 500 test giữ ở local. Manifest mô tả đủ 15.000 ảnh.

START_TRAINING=True và ALLOW_UNREVIEWED_SMOKE=True theo ủy quyền người dùng: chỉ smoke 1 epoch trên 5% train khi chưa review. Full visual review toàn bộ chưa xong: notebook cho chạy preflight nhưng chặn full run; chỉ cho phép smoke đã được ủy quyền. Sau review phải cập nhật release, hash và gói upload.

25 epoch mới, patience 10, imgsz 1536, tổng batch 16, cache=False. Fine-tune resume=False; lịch chạy trước [50,20], reset optimizer cho lượt mới. 8 ID giữ nguyên, ID 7 bottle gồm cup+bottle. Batch 32 ở imgsz 1536 đã CUDA OOM trên 2×T4; người dùng chọn batch 16.

Input nhanh: upload `project8_v05_trainval_best_v04_20261008.zip.bin` nguyên dạng. Dataset phải hiển thị **1 file**. Notebook kiểm SHA256 archive, giải nén vào `/tmp`, rồi kiểm toàn bộ ảnh/nhãn; giữ nguyên gate smoke 1 epoch/5%. Không gắn đồng thời dataset 29.005 file vào phiên mới.


In [ ]:
# 1. Cấu hình lần tiếp theo — chạy các cell theo thứ tự
START_TRAINING = True
ALLOW_UNREVIEWED_SMOKE = True  # user authorized smoke before full review on 2026-10-08
ARCHIVE_INPUT = ''  # tự tìm archive .zip.bin; để trống khi dùng dataset đã bung
DATASET_INPUT = ''  # để trống: tự nhận đúng manifest 15k trong /kaggle/input
CHECKPOINT_INPUT = ''  # để trống: chọn file .pt có đúng SHA256 best v0.4
GPU_IDS = 'auto'  # 'auto': dùng tối đa 2 GPU; hoặc '0' / '0,1'
EPOCHS = 25
PATIENCE = 10
IMGSZ = 1536
BATCH = 16  # tổng batch; 1536/32 đã CUDA OOM trên 2×T4
CLOSE_MOSAIC = 5
SMOKE_ONLY = True  # chỉ smoke hai cấu hình theo yêu cầu 09/10/2026
RUN_ONE_EPOCH_SMOKE = True  # khi bật train: kiểm 1 epoch trên 5% dữ liệu trước full run
OUTPUT_ROOT = '/kaggle/working/BTL-AI-Camera-v05'
OLD_NAMES = ['person', 'table', 'chair', 'laptop', 'cell phone', 'backpack', 'book', 'cup']
NAMES = OLD_NAMES[:-1] + ['bottle']
EXPECTED_IMAGES = {'train': 12000, 'val': 2500, 'test': 500}
EXPECTED_BOXES = {'train': 33272, 'val': 8359, 'test': 1645}
BEST_SHA256 = 'd6fa68a79abd7eebfc91724c683d2cfd0c2ace6f6623a019fbef44729bf47adb'
MANIFEST_SHA256 = '22a0140ab5e4d454d7e0c46ab24145c6d52d8b949db67ab8142fdd3bb8bb9ccd'
CHECKSUMS_SHA256 = '53a664318ad67bb64f6c894c21e94a6543382b5b9cf859e5ca9d4e283db4a77e'
PORTABLE_YAML_SHA256 = '4b26114dfaf683ce88b3b159cdec3f37f30f9b8168fd260c6cc1a977d718a122'


In [ ]:
# 2. Giữ phiên bản Ultralytics đã dùng tạo checkpoint; không cài requirements CPU local
import subprocess, sys, torch
before_torch = (torch.__version__, torch.version.cuda)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.165'], check=True)
import ultralytics, yaml
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from pathlib import Path
import csv, json, hashlib, math, shutil, datetime, gc, zipfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
assert ultralytics.__version__ == '8.4.165'
assert before_torch == (torch.__version__, torch.version.cuda), 'Torch/CUDA bị thay đổi; cần reset môi trường.'
print('Ultralytics:', ultralytics.__version__, '| Torch/CUDA:', before_torch)
print('GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 3. Nhận đúng bộ 15k và checkpoint best v0.4; không dùng best cũ chỉ vì cùng tên
INPUT_ROOT = Path('/kaggle/input')
def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def safe_path(base, relative):
    path = Path(relative)
    if path.is_absolute() or '..' in path.parts:
        raise ValueError(f'Đường dẫn không an toàn: {relative}')
    result = (base / path).resolve()
    if not result.is_relative_to(base.resolve()):
        raise ValueError(f'Đường dẫn ra ngoài dataset: {relative}')
    return result

# Một file Input; giữ đuôi .zip.bin để Kaggle không tự bung lúc upload.
import stat, time
ARCHIVE_SHA256 = '236972ca3e6142d977a29068cf217dc5312c8fb870c8094a98c4960386a0eb5b'
ARCHIVE_NAME = 'project8_v05_trainval_best_v04_20261008.zip.bin'
ARCHIVE_ROOT = None
archive_paths = [Path(ARCHIVE_INPUT)] if ARCHIVE_INPUT else sorted(INPUT_ROOT.rglob(ARCHIVE_NAME))
if len(archive_paths) > 1:
    raise RuntimeError('Có nhiều archive; đặt ARCHIVE_INPUT cho một file cụ thể.')
if archive_paths:
    archive_path = archive_paths[0]
    print('ARCHIVE_HASH_START:', archive_path, flush=True)
    started = time.perf_counter()
    assert file_sha(archive_path) == ARCHIVE_SHA256, 'Sai SHA256 archive train/val + BEST v0.4'
    ARCHIVE_ROOT = Path('/tmp/project8_v05_15k_runtime') / ('archive_' + ARCHIVE_SHA256[:16])
    ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        infos = archive.infolist()
        member_names = [info.filename for info in infos]
        assert len(member_names) == len(set(member_names)) == 29005, 'Archive thiếu/trùng file'
        manifest_bytes = archive.read('project8_v05/manifest.csv')
        assert hashlib.sha256(manifest_bytes).hexdigest() == MANIFEST_SHA256
        archive_rows = list(csv.DictReader(manifest_bytes.decode('utf-8-sig').splitlines()))
        assert len(archive_rows) == 15000
        expected_members = {
            'project8_v05/' + row[key]
            for row in archive_rows if row['split'] in {'train', 'val'}
            for key in ('image_path', 'label_path')
        } | {
            'project8_v05/manifest.csv', 'project8_v05/checksums.json',
            'project8_v05/data.yaml', 'project8_v05/training_readiness.json',
            'BEST_Project8_v04_20261008.pt',
        }
        assert set(member_names) == expected_members, 'Archive có file dư/thiếu hoặc test media'
        assert shutil.disk_usage(ARCHIVE_ROOT).free > sum(info.file_size for info in infos) + 512*1024**2, 'Thiếu ổ tạm để giải nén'
        # Kiểm toàn bộ đường dẫn trước khi ghi bất kỳ file nào.
        for info in infos:
            assert '\\' not in info.filename and not info.is_dir()
            assert not stat.S_ISLNK(info.external_attr >> 16), 'Không nhận symlink'
            safe_path(ARCHIVE_ROOT, info.filename)
        print('ARCHIVE_HASH_PASS: 29005 files; extract locally', flush=True)
        for index, info in enumerate(infos, 1):
            destination = safe_path(ARCHIVE_ROOT, info.filename)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as read, destination.open('wb') as write:
                shutil.copyfileobj(read, write, length=1024*1024)
            assert destination.stat().st_size == info.file_size
            if index % 2000 == 0 or index == len(infos):
                print('UNZIP:', index, '/', len(infos), '| seconds:', round(time.perf_counter()-started, 1), flush=True)
    INPUT_ROOT = ARCHIVE_ROOT
    print('ARCHIVE_READY:', INPUT_ROOT, '| seconds:', round(time.perf_counter()-started, 1), flush=True)
else:
    print('Không có archive; dùng Input đã bung sẵn.', flush=True)

manifests = [Path(DATASET_INPUT) / 'manifest.csv'] if DATASET_INPUT else sorted(INPUT_ROOT.rglob('manifest.csv'))
valid_roots = [p.parent for p in manifests if p.is_file() and file_sha(p) == MANIFEST_SHA256 and (p.parent/'images/train').is_dir()]
if not valid_roots:
    raise FileNotFoundError('Chưa gắn đúng dataset v0.5 15k. Add Input dataset một file .zip.bin hoặc dataset v0.5 đã bung sẵn.')
if len(valid_roots) != 1:
    raise RuntimeError('Có nhiều bản dataset 15k; đặt DATASET_INPUT cho một bản cụ thể.')
SOURCE_DATA = valid_roots[0]
assert file_sha(SOURCE_DATA / 'checksums.json') == CHECKSUMS_SHA256
checksums = json.loads((SOURCE_DATA / 'checksums.json').read_text(encoding='utf-8'))
with (SOURCE_DATA / 'manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
    rows = list(csv.DictReader(stream))
assert len(rows) == 15000 and Counter(r['split'] for r in rows) == EXPECTED_IMAGES
cfg = yaml.safe_load((SOURCE_DATA/'data.yaml').read_text(encoding='utf-8'))
names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][i] for i in range(8)]
assert names == NAMES and cfg['nc'] == 8
assert len({r['pixel_sha256'] for r in rows}) == 15000
expected_files = {r[k] for r in rows for k in ('image_path','label_path')}
assert set(checksums) == expected_files | {'manifest.csv','data.yaml'} and len(checksums) == 30002
assert checksums['manifest.csv'] == MANIFEST_SHA256
# Export YAML is portable; source data.yaml digest remains in the historical release checksums.
assert file_sha(SOURCE_DATA/'data.yaml') == PORTABLE_YAML_SHA256
assert cfg.get('train')=='images/train' and cfg.get('val')=='images/val' and not cfg.get('test')

weight_paths = [Path(CHECKPOINT_INPUT)] if CHECKPOINT_INPUT else sorted(INPUT_ROOT.rglob('*.pt'))
matched_weights = [p for p in weight_paths if p.is_file() and file_sha(p) == BEST_SHA256]
if not matched_weights:
    raise FileNotFoundError('Thiếu BEST_Project8_v04_20261008.pt của run v0.4. Upload/gắn file đúng hash; best của 30 epoch không được dùng.')
initial = matched_weights[0]  # các bản còn lại, nếu có, đều cùng byte/hash
model_check = YOLO(str(initial))
assert model_check.task == 'detect' and [model_check.names[i] for i in range(8)] == NAMES
print('Dataset:', SOURCE_DATA, '| split:', dict(Counter(r['split'] for r in rows)))
print('BEST v0.4:', initial, '| SHA256:', file_sha(initial))
print('ID 7 bottle gồm cup+bottle; giữ nguyên 8 ID từ best v0.4.')
del model_check; gc.collect()
readiness = json.loads((SOURCE_DATA/'training_readiness.json').read_text(encoding='utf-8'))
assert readiness['manifest_sha256'] == MANIFEST_SHA256
TRAIN_GATE = (readiness.get('training_ready') is True and readiness.get('full_visual_review_complete') is True and all(r['review_stage']=='full_review_complete' for r in rows))
print('Full visual review gate:', TRAIN_GATE)

assert {split:sum(int(r['boxes']) for r in rows if r['split']==split) for split in EXPECTED_IMAGES} == EXPECTED_BOXES


In [ ]:
# 4. Stage và kiểm SHA256/tọa độ toàn bộ train+val; test không vào thư mục train
WORK = Path('/tmp/project8_v05_15k_runtime')
DATA_DIR = SOURCE_DATA if ARCHIVE_ROOT is not None else WORK / 'dataset'
DATA_DIR.mkdir(parents=True, exist_ok=True)
train_val = [r for r in rows if r['split'] in {'train','val'}]

def stage_record(row):
    for key, hash_key in [('image_path','image_sha256'),('label_path','label_sha256')]:
        src = safe_path(SOURCE_DATA, row[key]); dst = safe_path(DATA_DIR, row[key])
        assert checksums[row[key]] == row[hash_key]
        assert file_sha(src) == row[hash_key], f'File nguồn bị đổi: {src}'
        if src != dst:
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.is_file() or file_sha(dst) != row[hash_key]:
                # Input đã bung: đọc/ghi thường để tránh sendfile trên mount.
                with src.open('rb') as read, dst.open('wb') as write:
                    shutil.copyfileobj(read, write, length=1024*1024)
            assert file_sha(dst) == row[hash_key], dst
        # Archive: file đã trên ổ tạm và vừa được hash; không chép/hash lần hai.
    counts = Counter()
    for line in (DATA_DIR/row['label_path']).read_text(encoding='utf-8').splitlines():
        values = list(map(float, line.split()))
        assert len(values)==5 and all(math.isfinite(v) for v in values)
        cls,x,y,w,h = values
        assert cls.is_integer() and 0<=cls<8 and 0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1
        counts[int(cls)] += 1
    assert sum(counts.values()) == int(row['boxes']) and counts
    return counts

class_counts = Counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    for i, counts in enumerate(pool.map(stage_record, train_val), 1):
        class_counts.update(counts)
        if i % 2000 == 0 or i == len(train_val):
            print('Stage/hash/labels:', i, '/', len(train_val), flush=True)
assert sum(class_counts.values()) == 41631
assert not (DATA_DIR/'images/test').exists() and not (DATA_DIR/'labels/test').exists()
actual_files = {p.relative_to(DATA_DIR).as_posix() for folder in ['images','labels'] for p in (DATA_DIR/folder).rglob('*') if p.is_file() and not p.name.endswith('.cache')}
assert actual_files == {r[k] for r in train_val for k in ('image_path','label_path')}, 'Có file dư trong scratch; dùng session mới.'
DATA_YAML = WORK/'training_data.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path':str(DATA_DIR.resolve()), 'train':'images/train', 'val':'images/val', 'nc':8, 'names':dict(enumerate(NAMES))},sort_keys=False),encoding='utf-8')
resolved = check_det_dataset(str(DATA_YAML),autodownload=False)
assert not resolved.get('test')
for split in ['train','val']:
    assert Path(resolved[split]) == (DATA_DIR/'images'/split).resolve()
OUT = Path(OUTPUT_ROOT); OUT.mkdir(parents=True, exist_ok=True)
PREPARED = {'dataset_manifest_sha256':MANIFEST_SHA256,'checkpoint_sha256':BEST_SHA256,'all_input_images':EXPECTED_IMAGES,'training_images':len(train_val),'training_boxes':41631,'classes':NAMES,'class_id_7':'cups+bottles by user request','resume':False,'cls_remap':False}
(OUT/'prepared.json').write_text(json.dumps(PREPARED,indent=2),encoding='utf-8')
print('PREPARED: 12.000 train + 2.500 val; 500 test giữ riêng. Chưa train.')
print('Box train+val:', {NAMES[i]:class_counts[i] for i in range(8)})

In [ ]:
# 5. Xem ảnh và bounding box của từng lớp trước train
from PIL import Image, ImageDraw
from IPython.display import display
canvas = Image.new('RGB',(4*320,2*260),'#eeeeee')
for cls,name in enumerate(NAMES):
    row = next(r for r in train_val if any(int(line.split()[0])==cls for line in (DATA_DIR/r['label_path']).read_text().splitlines()))
    with Image.open(DATA_DIR/row['image_path']) as im:
        im = im.convert('RGB'); im.thumbnail((312,220))
        draw = ImageDraw.Draw(im)
        for line in (DATA_DIR/row['label_path']).read_text().splitlines():
            c,x,y,w,h = map(float,line.split())
            if int(c)==cls:
                draw.rectangle(((x-w/2)*im.width,(y-h/2)*im.height,(x+w/2)*im.width,(y+h/2)*im.height),outline='lime',width=2)
        x0,y0 = (cls%4)*320,(cls//4)*260
        canvas.paste(im,(x0+(320-im.width)//2,y0))
        ImageDraw.Draw(canvas).text((x0+4,y0+228),f'{cls}: {name}',fill='black')
preview = OUT/'dataset_boxes_preview.jpg'; canvas.save(preview,quality=90)
display(canvas)

In [ ]:
# 6. Fine-tune GPU: MuSGD/lr0=.001; giữ numeric head rows khi cup đổi tên bottle
TRAIN_COMPLETED = False
RUN_DIR = None
if not START_TRAINING:
    print('Đã chuẩn bị. Đặt START_TRAINING=True ở cell 1 khi sẵn sàng, rồi chạy lại theo thứ tự.')
else:
    if not TRAIN_GATE and not ALLOW_UNREVIEWED_SMOKE:
        raise RuntimeError('Chưa hoàn tất full visual review 15.000 ảnh. Không được train; cập nhật release và review evidence trước.')
    if not torch.cuda.is_available():
        raise RuntimeError('Cần GPU CUDA trong Settings > Accelerator; không fallback sang CPU.')
    count = torch.cuda.device_count()
    ids = list(range(min(count,2))) if GPU_IDS=='auto' else [int(x) for x in GPU_IDS.split(',')]
    assert ids and len(set(ids))==len(ids) and all(0<=x<count for x in ids)
    device = ','.join(map(str,ids))
    batch = BATCH if BATCH is not None else 8*len(ids)
    assert isinstance(batch,int) and batch>=len(ids) and batch%len(ids)==0
    assert isinstance(EPOCHS,int) and EPOCHS>=10 and PATIENCE>0 and 0<=CLOSE_MOSAIC<EPOCHS
    assert file_sha(initial)==BEST_SHA256
    yolo_cli = shutil.which('yolo')
    if not yolo_cli:
        raise RuntimeError('Không tìm thấy yolo CLI; chạy lại cell cài Ultralytics.')
    # CLI subprocess để DDP chạy trong script entrypoint, không chạy DDP trực tiếp trong notebook.
    stamp = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).strftime('%Y%m%d_%H%M%S')
    run_name = f'project8_v05_15k_from_best_v04_{stamp}'
    RUNS = OUT/'kaggle_runs'
    train_config = dict(model=str(initial),data=str(DATA_YAML),project=str(RUNS),name=run_name,
                        epochs=EPOCHS,patience=PATIENCE,imgsz=IMGSZ,batch=batch,device=device,
                        workers=2,cache=False,fraction=1.0,optimizer='MuSGD',lr0=0.001,
                        seed=42,deterministic=True,mosaic=0.5,mixup=0.0,cutmix=0.0,
                        close_mosaic=CLOSE_MOSAIC,save=True,save_period=1,resume=False,
                        cls_remap=False,val=True,nms=False,plots=True,exist_ok=False)
    def run_cli(config,filename):
        config_path = OUT/filename
        config_path.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
        command = [yolo_cli,'detect','train',f'cfg={config_path}']
        print('Chạy:',command,'| GPUs:',device,'| batch:',batch,flush=True)
        subprocess.run(command,check=True,cwd=str(OUT))
    if RUN_ONE_EPOCH_SMOKE or not TRAIN_GATE:
        smoke = dict(train_config,epochs=1,fraction=0.05,close_mosaic=0,name=run_name+'_smoke')
        run_cli(smoke,'smoke_config.yaml')
        plots = sorted((RUNS/(run_name+'_smoke')).glob('train_batch*.jpg'))
        if not plots:
            raise RuntimeError('Smoke không tạo train_batch: kiểm tra dataset trước full run.')
        display(Image.open(plots[0]))
    if TRAIN_GATE and not SMOKE_ONLY:
        run_cli(train_config,'train_config.yaml')
        RUN_DIR = RUNS/run_name
        best = RUN_DIR/'weights/best.pt'; last = RUN_DIR/'weights/last.pt'
        assert best.is_file() and last.is_file()
        TRAIN_COMPLETED = True
        print('Full run xong. Best:',best,'| SHA256:',file_sha(best))
    else:
        smoke_dir = RUNS/(run_name+'_smoke')
        smoke_result = dict(PREPARED,status='smoke_only',epochs=1,fraction=0.05,full_visual_review_complete=bool(TRAIN_GATE),full_training_completed=False,best_sha256=file_sha(smoke_dir/'weights/best.pt'))
        (OUT/'smoke_result.json').write_text(json.dumps(smoke_result,indent=2),encoding='utf-8')
        print('SMOKE_ONLY_COMPLETE: 1 epoch, 5% train; full_visual_review_complete:', TRAIN_GATE, '; no full run.')

In [ ]:
# So sánh hai smoke: cùng BEST v0.4, 600 train + 2500 val; không full train.
assert not TRAIN_COMPLETED, 'Cell này chỉ so sánh smoke; không dùng kết quả full run'
FIRST_SMOKE_DIR = smoke_dir
first_args = yaml.safe_load((FIRST_SMOKE_DIR/'args.yaml').read_text())
assert first_args['imgsz'] == 1536 and first_args['batch'] == 16
assert file_sha(initial) == BEST_SHA256
second_name = run_name + '_1280_batch32_smoke'
second_config = dict(smoke, model=str(initial), imgsz=1280, batch=32, epochs=1,
                     fraction=0.05, close_mosaic=0, name=second_name)
second_config_path = OUT/'smoke_1280_batch32_config.yaml'
second_config_path.write_text(yaml.safe_dump(second_config,sort_keys=False),encoding='utf-8')
print('SECOND_SMOKE_START: imgsz=1280, batch=32, epochs=1, fraction=0.05, GPUs=0,1',flush=True)
subprocess.run([yolo_cli,'detect','train',f'cfg={second_config_path}'],check=True,cwd=str(OUT))
SECOND_SMOKE_DIR = RUNS/second_name
assert (SECOND_SMOKE_DIR/'weights/best.pt').is_file()
protocol = dict(data=str(DATA_YAML), split='val', imgsz=1536, batch=8, device=0,
                conf=0.001, iou=0.7, nms=True, max_det=300, augment=False,
                workers=2, plots=True, quantize=16)
comparison = {'status':'PASS','scope':'two one-epoch smoke runs; no full training',
              'initial_sha256':BEST_SHA256,'manifest_sha256':MANIFEST_SHA256,
              'training_images_per_smoke':600,'validation_images':2500,
              'full_visual_review_complete':bool(TRAIN_GATE),'protocol':protocol,'runs':{}}
COMPARE_DIR = OUT/'smoke_comparison'
COMPARE_DIR.mkdir(parents=True,exist_ok=True)
for key, directory in [('1536_batch16',FIRST_SMOKE_DIR),('1280_batch32',SECOND_SMOKE_DIR)]:
    records = list(csv.DictReader((directory/'results.csv').open(encoding='utf-8')))
    assert len(records)==1 and all(math.isfinite(float(v)) for v in records[0].values())
    checkpoint = directory/'weights/best.pt'
    candidate = YOLO(str(checkpoint))
    assert candidate.task=='detect' and [candidate.names[i] for i in range(8)] == NAMES
    print('COMMON_VAL_START:',key,'| imgsz=1536, batch=8, GPU=0',flush=True)
    measured = candidate.val(**protocol,project=str(COMPARE_DIR),name=key,exist_ok=False)
    per_class = {}
    for j, cls in enumerate(measured.box.ap_class_index):
        per_class[NAMES[int(cls)]] = {'precision':float(measured.box.p[j]),
            'recall':float(measured.box.r[j]),'mAP50':float(measured.box.ap50[j]),
            'mAP50_95':float(measured.box.ap[j])}
    record = {'directory':str(directory),'best_sha256':file_sha(checkpoint),
              'epoch_seconds_train_plus_val':float(records[0]['time']),
              'precision':float(measured.box.mp),'recall':float(measured.box.mr),
              'mAP50':float(measured.box.map50),'mAP50_95':float(measured.box.map),
              'speed_ms_per_image':{k:float(v) for k,v in measured.speed.items()},
              'per_class':per_class}
    comparison['runs'][key] = record
    (COMPARE_DIR/(key+'_metrics.json')).write_text(json.dumps(record,indent=2),encoding='utf-8')
    print('COMMON_VAL_RESULT',key,json.dumps(record),flush=True)
    del candidate, measured
    gc.collect(); torch.cuda.empty_cache()
(OUT/'smoke_comparison.json').write_text(json.dumps(comparison,indent=2),encoding='utf-8')
print('TWO_SMOKE_COMPARISON_PASS',json.dumps(comparison),flush=True)
import pandas as pd
summary = []
for key,r in comparison['runs'].items():
    summary.append({'Config':key,'Epoch train+val (s)':round(r['epoch_seconds_train_plus_val'],2),
                    'Precision (%)':round(100*r['precision'],2),'Recall (%)':round(100*r['recall'],2),
                    'mAP50 (%)':round(100*r['mAP50'],2),'mAP50-95 (%)':round(100*r['mAP50_95'],2)})
display(pd.DataFrame(summary).set_index('Config').T)
print('Raw timing from sequential validators is not a controlled inference FPS comparison.')


In [ ]:
# 7. Chấm BEST cũ và BEST mới trên cùng validation 15k; không dùng test
if not TRAIN_COMPLETED:
    if 'comparison' in globals() and len(comparison['runs']) == 2:
        import pandas as pd
        summary = {}
        for key, r in comparison['runs'].items():
            summary[key] = {'Epoch train+val (s)':round(r['epoch_seconds_train_plus_val'],2),
                            'Precision (%)':round(100*r['precision'],2),
                            'Recall (%)':round(100*r['recall'],2),
                            'mAP50 (%)':round(100*r['mAP50'],2),
                            'mAP50-95 (%)':round(100*r['mAP50_95'],2)}
        display(pd.DataFrame(summary))
        per_class = {name:{key:round(100*r['per_class'][name]['mAP50_95'],2)
                          for key,r in comparison['runs'].items()} for name in NAMES}
        print('mAP50-95 (%) từng lớp — cùng val 1536; 1 epoch/600 train, chưa full review:')
        display(pd.DataFrame.from_dict(per_class,orient='index'))
        print('Hai smoke đã xong. Không chạy 25 epoch. Không dùng raw validator timing để kết luận FPS.')
    else:
        print('Chưa có kết quả hai smoke; chạy cell so sánh trước.')
else:
    EVAL_PROTOCOL = dict(data=str(DATA_YAML),split='val',imgsz=640,batch=8,device=ids[0],
                         conf=0.001,iou=0.7,nms=True,max_det=300,augment=False,workers=2,plots=True)
    def evaluate_checkpoint(checkpoint,label):
        evaluated = YOLO(str(checkpoint))
        assert evaluated.task=='detect' and len(evaluated.names)==8
        evaluated.model.names = dict(enumerate(NAMES))  # tên báo cáo; không thay numeric ID hoặc tensor
        metrics = evaluated.val(**EVAL_PROTOCOL,project=str(RUN_DIR/'comparison'),name=label,exist_ok=False)
        record = {'checkpoint_sha256':file_sha(checkpoint),'dataset_manifest_sha256':MANIFEST_SHA256,
                  'protocol':EVAL_PROTOCOL,'mAP50':float(metrics.box.map50),'mAP50_95':float(metrics.box.map),
                  'per_class':{}}
        for j,cls in enumerate(metrics.box.ap_class_index):
            record['per_class'][NAMES[int(cls)]] = {'AP50':float(metrics.box.ap50[j]),'AP50_95':float(metrics.box.ap[j])}
        (RUN_DIR/f'{label}_val_metrics.json').write_text(json.dumps(record,indent=2),encoding='utf-8')
        del evaluated; gc.collect(); torch.cuda.empty_cache()
        return record
    before = evaluate_checkpoint(initial,'best_v04_before')
    after = evaluate_checkpoint(best,'best_v05_after')
    print('mAP50 cũ/mới trên CÙNG val mới:',before['mAP50'],after['mAP50'])
    print('mAP50-95 cũ/mới:',before['mAP50_95'],after['mAP50_95'])
    for name in NAMES:
        print(name, before['per_class'].get(name), '->', after['per_class'].get(name))

In [ ]:
# 8. Đóng ZIP kết quả: best/last, CSV, cấu hình, metadata và cả 2 phép chấm
if not TRAIN_COMPLETED:
    print('Chưa train: không có result ZIP.')
else:
    result = dict(PREPARED,status='completed',run_id=RUN_DIR.name,
                  parent_history_epochs=[50,20],full_run_epochs=len(list(csv.DictReader((RUN_DIR/'results.csv').open()))),
                  smoke_epoch_separate=bool(RUN_ONE_EPOCH_SMOKE),best_sha256=file_sha(best),
                  last_sha256=file_sha(last),ultralytics=ultralytics.__version__,torch=torch.__version__)
    (RUN_DIR/'result.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
    sources = {'best.pt':best,'last.pt':last,'results.csv':RUN_DIR/'results.csv',
               'args.yaml':RUN_DIR/'args.yaml','result.json':RUN_DIR/'result.json',
               'best_v04_before_val_metrics.json':RUN_DIR/'best_v04_before_val_metrics.json',
               'best_v05_after_val_metrics.json':RUN_DIR/'best_v05_after_val_metrics.json',
               'prepared.json':OUT/'prepared.json','dataset_boxes_preview.jpg':preview}
    assert all(p.is_file() for p in sources.values()), 'Thiếu kết quả; chạy cell đánh giá trước.'
    files = {name:{'sha256':file_sha(p),'bytes':p.stat().st_size} for name,p in sources.items()}
    result_zip = OUT/(RUN_DIR.name+'_result.zip')
    with zipfile.ZipFile(result_zip,'w',zipfile.ZIP_DEFLATED) as archive:
        for name,p in sources.items(): archive.write(p,name)
        archive.writestr('FILES.json',json.dumps({'schema_version':2,'files':files},indent=2))
    with zipfile.ZipFile(result_zip) as archive:
        assert archive.testzip() is None
    print('Result ZIP:',result_zip,'| SHA256:',file_sha(result_zip))
    print('Dùng Save Version để lưu output hoặc tải ZIP về trước khi session kết thúc.')

## Trước khi train

Run All sẽ chạy preflight và smoke 1 epoch trên 5% train theo ủy quyền; hoàn thành full review và phát hành lại evidence/hash trước khi bật train. Dùng GPU CUDA, kiểm smoke 1 epoch riêng trước full run. So sánh best cũ/mới trên cùng 2.500 validation; không dùng 500 test để chọn model.
